In [1]:
import numpy as np

def classify_contcar_geometry(filepath, bond_length_N2=1.1, tol=0.2):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    scale = float(lines[1])
    lattice = np.array([list(map(float, lines[i].split())) for i in range(2, 5)]) * scale

    atom_counts = list(map(int, lines[6].split()))
    n_Pd = atom_counts[0]
    n_N = atom_counts[1]
    n_atoms = n_Pd + n_N

    coord_start = 9
    coords_direct = np.array([
        list(map(float, lines[i].split()[:3]))
        for i in range(coord_start, coord_start + n_atoms)
    ])
    coords_cartesian = coords_direct @ lattice  # Convert to Cartesian

    # Extract last two atoms: N atoms
    N1, N2 = coords_cartesian[-2], coords_cartesian[-1]
    N_distance = np.linalg.norm(N2 - N1)

    # Determine top-layer Pd z
    z_Pd_top = np.max(coords_cartesian[:n_Pd, 2])
    z_N_center = np.mean([N1[2], N2[2]])

    # Classification logic
    bond_min = bond_length_N2 - tol
    bond_max = bond_length_N2 + tol

    if N_distance > bond_max:
        return 'dissociation'
    elif bond_min <= N_distance <= bond_max:
        if z_N_center - z_Pd_top > 5:
            return 'bouncing back'
        elif z_N_center - z_Pd_top < 2:
            return 'adsorbed'

    return 'unclassified'

In [11]:
import numpy as np

def classify_contcar_geometry(filepath, bond_length_N2=1.1, tol=0.2):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    # Parse lattice vectors
    scale = float(lines[1])
    lattice = np.array([list(map(float, lines[i].split())) for i in range(2, 5)]) * scale

    # Atom counts
    atom_counts = list(map(int, lines[6].split()))
    n_Pd = atom_counts[0]
    n_N = atom_counts[1]
    n_atoms = n_Pd + n_N

    # Read direct coordinates and convert to Cartesian
    coord_start = 9
    coords_direct = np.array([
        list(map(float, lines[i].split()[:3]))
        for i in range(coord_start, coord_start + n_atoms)
    ])
    coords_cartesian = coords_direct @ lattice

    # Extract nitrogen atoms (last two)
    N1, N2 = coords_cartesian[-2], coords_cartesian[-1]
    N_distance = np.linalg.norm(N2 - N1)

    # Get top Pd z-position and N2 center height
    z_Pd_top = np.max(coords_cartesian[:n_Pd, 2])
    z_N_center = np.mean([N1[2], N2[2]])

    # Thresholds
    bond_min = bond_length_N2 - tol
    bond_max = bond_length_N2 + tol

    # Corrected classification logic
    if z_N_center - z_Pd_top < 2:
        if N_distance > bond_max:
            return 'dissociation'
        else:
            return 'adsorbed'
    elif bond_min <= N_distance <= bond_max and z_N_center - z_Pd_top > 5:
        return 'bouncing back'

    return 'bouncing back'

In [12]:
result = classify_contcar_geometry('CONTCAR_54')
print("Classification:", result)

Classification: dissociation


In [13]:
result = classify_contcar_geometry('CONTCAR_1')
print("Classification:", result)

Classification: bouncing back
